In [3]:
from nl_vlm import PropGeom, PropMesh, Vehicle, VLM, WindField
from nl_vlm.dynamics.rotor import rotate_vehicle_mesh
from nl_vlm.preprocess import generate_polars
from nl_vlm.reporting import SimReporter

import numpy as np
import pyvista as pv

# ------------------------------- parameters -------------------------------
rho = 1.225                 # density [kg/m^3]
rpm = 570                   # rotational speed [rpm]
dt = 0.75
R_tip = 2.809               # tip radius [m]
R_hub = 0.3354
num_blades = 3
J = 0.00                    # advance ratio; 0 = hover
arm_length = 1.35           # rotor arm (in tip-radii) for the quad hub layout

# --- atmosphere (feeds the polar Re/Mach) ---
mu = 1.81e-5                # dynamic viscosity [Pa*s]
a_sound = 343.0             # speed of sound [m/s]

# --- XFOIL alpha sweep (polar generation only) ---
alpha_i = -10.0             # start alpha [deg]
alpha_f = 20.0              # end alpha [deg]
alpha_step = 0.25           # alpha increment [deg]

com_position = np.array([356.4102564102564, 134.6153846153846, 50])
initial_body_velocity = np.array([0, 0, 0])
initial_euler_attitude = np.array([0, 0, 0])   # (roll, pitch, yaw) [rad]

# --------------------------------- paths ----------------------------------
BLADE_DIR   = '../data/blades/nasa_uam_quadrotor'
AIRFOIL_DIR = '../data/airfoils/nasa_uam_quadrotor'
POLAR_DIR   = '../data/polars/nasa_uam_quadrotor'
XFOIL_EXE   = '../tools/xfoil/xfoil.exe'   # not installed with nl_vlm; vendored here
WIND_VTK    = '../wind/vtk_resampled/resampled_129.vtk'

airfoil_dist_file = f'{BLADE_DIR}/nasa_uam_quadrotor_airfoildist.csv'
chorddist_file    = f'{BLADE_DIR}/nasa_uam_quadrotor_chorddist.csv'
pitchdist_file    = f'{BLADE_DIR}/nasa_uam_quadrotor_pitchdist.csv'
sweepdist_file    = f'{BLADE_DIR}/nasa_uam_quadrotor_sweepdist.csv'
# this blade has no heightdist

# ----------------------------- polar couplings -----------------------------
#   nonlinear_lift : converge gamma against Cl_polar(alpha_eff) -> real lift slope + stall
nonlinear_lift = True


# --------------------------- polar generation ------------------------------
# Off by default: XFOIL solves every station, which takes a while. Flip to True to
# rebuild POLAR_DIR at the parameters above -- do that whenever rpm, R_tip, J or rho
# change, since Re and Mach follow from them.
REGENERATE_POLARS = False

if REGENERATE_POLARS:
    generate_polars(
        airfoil_dist_file=airfoil_dist_file,
        chorddist_file=chorddist_file,
        airfoil_dir=AIRFOIL_DIR,
        out_dir=POLAR_DIR,
        xfoil_exe=XFOIL_EXE,
        R_tip=R_tip, rpm=rpm, J=J,
        rho=rho, mu=mu, a_sound=a_sound,
        alpha_i=alpha_i, alpha_f=alpha_f, alpha_step=alpha_step,
    )

# ------------------------------- wind field -------------------------------
# distributed urban wind sampled per control point (pass wind_field=None for still air)
wind_field = WindField(pv.read(WIND_VTK))

# ------------------------------- geometry / mesh -------------------------------
geometry = PropGeom(
    airfoil_distribution_file=airfoil_dist_file,
    chorddist_file=chorddist_file,
    pitchdist_file=pitchdist_file,
    sweepdist_file=sweepdist_file,
    airfoil_path=AIRFOIL_DIR,
    polar_path=POLAR_DIR,
    R_tip=R_tip, R_hub=R_hub, num_blades=num_blades,
)
propeller_mesh = PropMesh(geometry, span_resolution=12, chord_resolution=5)

# quad layout: four hubs on the arms; spins alternate (+, -, -, +)
hubs = np.array([
    [ arm_length * R_tip,  arm_length * R_tip, 0.25 * R_tip],
    [ arm_length * R_tip, -arm_length * R_tip, 0.25 * R_tip],
    [-arm_length * R_tip,  arm_length * R_tip, 0.6  * R_tip],
    [-arm_length * R_tip, -arm_length * R_tip, 0.6  * R_tip],
]) + com_position


vehicle = Vehicle(propeller_mesh, hub_positions=hubs, spin_directions=[1, -1, -1, 1])
vehicle_mesh = vehicle.generate_vehicle()

omega_rad = rpm * 2 * np.pi / 60
magVinf = J * rpm / 60 * (2 * R_tip)
freestream = np.array([0, 0, -magVinf])
omega_dict = {
    'Propeller_1': np.array([0, 0,  omega_rad]),
    'Propeller_2': np.array([0, 0, -omega_rad]),
    'Propeller_3': np.array([0, 0, -omega_rad]),
    'Propeller_4': np.array([0, 0,  omega_rad]),
}

# ------------------------- time-stepping over azimuth -------------------------
n_rev      = 1                       # number of revolutions
n_step_rev = 1                       # steps per ONE revolution
total_steps = n_rev * n_step_rev     # total time steps
d_psi = 2 * np.pi / n_step_rev       # azimuth turned each step [rad]

forces, moments = [], []
solved_mesh = None

vlm_solver = VLM(vehicle_mesh, propeller_mesh,
                 nonlinear_lift=nonlinear_lift)

fm = forces_and_moments = vlm_solver.calculate_total_forces_and_moments(
    propeller_mesh=vehicle_mesh,
    dt=dt, rho=rho, time_step=1,
    body_velocity=initial_body_velocity,
    freestream=freestream,
    omega=omega_dict,
    wind_field=wind_field,
    com_position=com_position,
    euler_angles=initial_euler_attitude,
)

# solved_mesh = vehicle_mesh

total_moment = np.zeros(3)
for prop_key in fm:
    total_moment += fm[prop_key]['total_body_moment']
print('total body moment =', total_moment)




  NL-VLM  |  analysis starting
  Rotors        : 4
  Blades/rotor  : 3
  Resolution    : 12 span x 5 chord  (528 panels)
  Rotor speed   : 570 rpm  (59.7 rad/s)
  Air density   : 1.2250 kg/m^3
  Freestream    : [ 0.  0. -0.] m/s
--------------------------------------------------------
  solving Propeller_1  [########################] 100.0%   lift coupling converged in 794 iters   res 1.0e-06      
  solving Propeller_2  [########################] 100.0%   lift coupling converged in 692 iters   res 9.9e-07      
  solving Propeller_3  [########################] 100.0%   lift coupling converged in 755 iters   res 1.0e-06      
  solving Propeller_4  [########################] 100.0%   lift coupling converged in 706 iters   res 9.9e-07      
--------------------------------------------------------
  VLM solved
    Propeller_1:  thrust Fz = +3493.0075 N    torque Mz = -9.4058e+02 N*m
    Propeller_2:  thrust Fz = +4331.2583 N    torque Mz = +1.1347e+03 N*m
    Propeller_3:  thrust Fz = +3

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection

# --- Journal font settings ---
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.linewidth": 1.2
})

propeller_key = 'Propeller_1'
propeller_data = vehicle_mesh[propeller_key]

fig = plt.figure(figsize=(6, 3), dpi=300)
ax = fig.add_axes([0.1, 0.2, 0.8, 0.7])  # unchanged

hub_position = np.array(propeller_data['Hub Position'])
blade_keys = list(propeller_data['Blades'].keys())

all_pressures = [p for blade_key in blade_keys 
                 for p in propeller_data['Blades'][blade_key]['Pressure Difference'].values()]
max_abs_pressure = max(abs(min(all_pressures)), abs(max(all_pressures)))

patches = []
pressure_values = []

for blade_key in blade_keys:
    blade_data = propeller_data['Blades'][blade_key]
    for panel_index, panel in blade_data['Panels'].items():
        patches.append(Polygon([(-p[1], p[0]) for p in panel], closed=True))
        pressure = blade_data['Pressure Difference'][panel_index]
        pressure = max(pressure, 0)
        pressure_values.append(pressure / max_abs_pressure if max_abs_pressure != 0 else 0)

collection = PatchCollection(
    patches,
    cmap='viridis_r',
    alpha=0.85,
    edgecolors='face',
    linewidths=0.4,
    antialiased=True
)

collection.set_array(np.array(pressure_values))

vmin = -max(abs(min(pressure_values)), abs(max(pressure_values)))
collection.set_clim(vmin, -vmin)

ax.add_collection(collection)

# --- Colorbar ---
cax = fig.add_axes([0.1, 0.15, 0.8, 0.03])
cbar = plt.colorbar(collection, cax=cax, orientation='horizontal')
cbar.set_label('Pressure Difference')
cbar.ax.tick_params(length=4, width=1)
cbar.set_label('Pressure Difference', fontsize=8)
cbar.ax.tick_params(labelsize=8) 

all_points = np.array([(-p[1], p[0]) for blade_key in blade_keys
                       for panel in propeller_data['Blades'][blade_key]['Panels'].values()
                       for p in panel])

x_min, y_min = np.min(all_points, axis=0)
x_max, y_max = np.max(all_points, axis=0)
margin = 0.05

ax.set_xlim(x_min - margin*(x_max-x_min), x_max + margin*(x_max-x_min))
ax.set_ylim(y_min - margin*(y_max-y_min), y_max + margin*(y_max-y_min))
ax.set_aspect('equal')

ax.plot(-hub_position[1], hub_position[0], 'ko', markersize=5)

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()